In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import matplotlib.animation as animation
from matplotlib.collections import LineCollection
from matplotlib.colors import to_rgba

import os

from numba import njit

## Representation of the equations:

Suppose there are $n_s$ masses, in a $n_d$ dimensional space, we define the position matrix as $$q\in\mathbb{R}^{n_s \times n_d}$$
For a single mass, the position in space is represented with $$\bold q_i \in \mathbb{R}^{n_d}$$

The entire time series of the positions can be given as (for $N_i$ iterations):
$$q(t) \in \mathbb{R}^{(N_i+1) \times n_s \times n_d} $$

## Force Equation:

The acceleration of mass $i$ due to mass $j$ is given by the gravitational acceleration:

$$\ddot{\bold q}_i = \frac{\bold q_j - \bold q_i}{\left\vert\bold q_j - \bold q_i\right\vert^3}$$
Where we are working in units where $G = m =1$

In [ ]:
@njit
def symplectic_stromer_verlet(func, q0, v0, dt, N_i):
    # number of stars
    n_s = q0.shape[0]
    # number of dimensions (mostly 2, but supports 3)
    n_d = q0.shape[1]

    # allocate arrays to store the entire 
    q = np.empty((N_i + 1, n_s, n_d))
    v = np.empty((N_i + 1, n_s, n_d))
    t = dt * np.arange(N_i + 1)

    # set the initial states
    q[0] = q0
    v[0] = v0

    # main iteration loop
    for i in range(N_i):
        # calculate intermediate variables
        Q1 = q[i] + 0.5 * dt * v[i]
        f1 = func(Q1, t[i] + 0.5 * dt)

        # updation step
        v[i+1] = v[i] + dt * f1
        q[i+1] = q[i] + dt * v[i] + 0.5 * (dt ** 2) * f1

    # return the calculated parameters
    return t, q, v

In [ ]:
@njit
def acceleration(q, t):
    # q is current positions: shape (n_s, n_d)
    
    # pairwise displacement vectors: shape (n_s, n_s, n_d)
    disp = q[:, None, :] - q[None, :, :]

    # pairwise distances: shape (n_s, n_s)
    dist_sq = np.sum(disp ** 2, axis = -1)

    # adds a diagonal term to distances to prevent division by zero
    # doesn't affect the physics, since mass has zero force on itself
    dist_sq = dist_sq + np.identity(q.shape[0])

    # force on j-th mass, due the the i-th mass: (n_s, n_s, n_d)
    force_ij = disp / (dist_sq[:, :, None] ** 1.5)

    # total force on the i-th mass
    force = np.sum(force_ij, axis=0)

    return force

## The actual system

The system we are going to simulate is Moore's 3-body figure eight.

It is a very stable system of gravitationally attracting bodies, chasing each other periodically.

In [ ]:
q0 = np.array([
    [0.97000436, -0.24308753],
    [-0.97000436, 0.24308753],
    [0, 0]
])

v0 = np.array([
    [0.466203685, 0.43236573],
    [0.466203685, 0.43236573],
    [-0.93240737, -0.86473146]
])

In [ ]:
# time step and number of iterations: 3200 steps of 0.01 is ~5 periods of the figure eight (T ~ 6.33)
dt = 0.01
N_i = 3200

t, q, v = symplectic_stromer_verlet(acceleration, q0, v0, dt, N_i)

## Plots and animations for the presentation

Everything below uses a dark theme. Change `BG` / `FG` in the next cell to match the slide background exactly. Files are written to `figures/` (`.mp4` if `ffmpeg` is installed, otherwise `.gif`; set `SAVE_GIF = True` to always get gifs).

In [ ]:
# ---------- dark theme + shared helpers ----------
BG = "#0f111a"        # background (match to the slide colour)
FG = "#e6e6e6"        # text / axes
GRID = "#2a2e3d"
BODY_COLORS = ["#4fc3f7", "#ff8a65", "#aed581"]   # one colour per body
PERT_COLORS = ["#ce93d8", "#fff176", "#80cbc4"]   # for the perturbed trajectory
REF_COLOR = "#8a90a8"

plt.rcParams.update({
    "figure.facecolor": BG, "axes.facecolor": BG, "savefig.facecolor": BG,
    "axes.edgecolor": FG, "axes.labelcolor": FG, "text.color": FG,
    "xtick.color": FG, "ytick.color": FG, "grid.color": GRID,
    "legend.facecolor": BG, "legend.edgecolor": GRID,
    "font.size": 13, "figure.dpi": 100,
})

OUT_DIR = "figures"
os.makedirs(OUT_DIR, exist_ok=True)

SAVE_GIF = False  # also write .gif files (large and slow; used automatically if ffmpeg is missing)
FPS = 30        # frames per second of the animations
STRIDE = 4      # use every STRIDE-th simulation step as a frame
TRAIL = 300     # trail length (in simulation steps)


def fading_trail(ax, color, lw=2.2):
    """An (initially empty) line collection used as a fading trail."""
    lc = LineCollection([], linewidths=lw, capstyle="round")
    ax.add_collection(lc)
    return lc, np.array(to_rgba(color))


def update_trail(lc, base_rgba, pts):
    """pts: (m, 2) most recent positions of one body, oldest first."""
    if len(pts) < 2:
        lc.set_segments([])
        return
    segs = np.stack([pts[:-1], pts[1:]], axis=1)
    alpha = np.linspace(0.0, 1.0, len(segs)) ** 1.5
    rgba = np.tile(base_rgba, (len(segs), 1))
    rgba[:, 3] = alpha
    lc.set_segments(segs)
    lc.set_colors(rgba)


def setup_axes(ax, xlim, ylim, title=None):
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_aspect("equal")
    ax.set_xticks([])
    ax.set_yticks([])
    for s in ax.spines.values():
        s.set_visible(False)
    if title:
        ax.set_title(title, pad=10)


def save_anim(anim, name):
    path = os.path.join(OUT_DIR, name)
    has_ffmpeg = animation.writers.is_available("ffmpeg")
    if has_ffmpeg:
        anim.save(path + ".mp4", writer=animation.FFMpegWriter(fps=FPS, bitrate=4000))
    if SAVE_GIF or not has_ffmpeg:
        anim.save(path + ".gif", writer=animation.PillowWriter(fps=FPS))
    plt.close(anim._fig)


def hamiltonian(q, v):
    """H = kinetic + potential, for unit masses and G = 1. q, v: (N, n_s, n_d)."""
    kinetic = 0.5 * np.sum(v ** 2, axis=(1, 2))
    disp = q[:, :, None, :] - q[:, None, :, :]
    r = np.sqrt(np.sum(disp ** 2, axis=-1))
    i, j = np.triu_indices(q.shape[1], k=1)
    potential = -np.sum(1.0 / r[:, i, j], axis=1)
    return kinetic + potential

### A standard (non-symplectic) integrator for comparison

Explicit (forward) Euler, with the same signature as the Störmer–Verlet routine and the same time step. It is the simplest method that is *not* symplectic, and its error shows up within the first period.

In [ ]:
@njit
def explicit_euler(func, q0, v0, dt, N_i):
    n_s = q0.shape[0]
    n_d = q0.shape[1]

    q = np.empty((N_i + 1, n_s, n_d))
    v = np.empty((N_i + 1, n_s, n_d))
    t = dt * np.arange(N_i + 1)

    q[0] = q0
    v[0] = v0

    for i in range(N_i):
        f = func(q[i], t[i])
        q[i+1] = q[i] + dt * v[i]
        v[i+1] = v[i] + dt * f

    return t, q, v


# same initial condition, same time step, same number of steps
t_e, q_e, v_e = explicit_euler(acceleration, q0, v0, dt, N_i)

# a very fine symplectic run over ~1 period, used as a faint "true orbit" guide in the plots
_, q_ref, _ = symplectic_stromer_verlet(acceleration, q0, v0, 1e-4, 63260)

### 1. Main animation: the figure-eight

In [ ]:
frames = np.arange(0, N_i + 1, STRIDE)

xlim = (q[..., 0].min() - 0.2, q[..., 0].max() + 0.2)
ylim = (q[..., 1].min() - 0.2, q[..., 1].max() + 0.2)

fig, ax = plt.subplots(figsize=(8, 4.5))
setup_axes(ax, xlim, ylim)

# faint guide of the full orbit
for b in range(3):
    ax.plot(q_ref[:, b, 0], q_ref[:, b, 1], color=REF_COLOR, lw=0.8, alpha=0.25)

trails = [fading_trail(ax, BODY_COLORS[b]) for b in range(3)]
dots = [ax.plot([], [], "o", color=BODY_COLORS[b], ms=12, mec="white", mew=0.8)[0] for b in range(3)]
time_text = ax.text(0.02, 0.95, "", transform=ax.transAxes, va="top")


def update_main(k):
    for b in range(3):
        pts = q[max(0, k - TRAIL):k + 1, b]
        update_trail(*trails[b], pts)
        dots[b].set_data([q[k, b, 0]], [q[k, b, 1]])
    time_text.set_text(f"t = {t[k]:.1f}")
    return [tr[0] for tr in trails] + dots + [time_text]


anim_main = animation.FuncAnimation(fig, update_main, frames=frames, interval=1000 / FPS, blit=True)
save_anim(anim_main, "moore_main")

### 2. Symplectic vs. a standard integrator (same time step)

In [ ]:
lim = max(np.abs(q_e).max(), np.abs(q).max()) + 0.2
xlim_c, ylim_c = (-lim, lim), (-lim, lim)

fig, axes = plt.subplots(1, 2, figsize=(12, 5.6))
titles = ["Symplectic (Störmer–Verlet)", "Explicit Euler (non-symplectic)"]
data = [q, q_e]

trails_c, dots_c = [], []
for ax, title, Q in zip(axes, titles, data):
    setup_axes(ax, xlim_c, ylim_c, title)
    for b in range(3):
        ax.plot(q_ref[:, b, 0], q_ref[:, b, 1], color=REF_COLOR, lw=0.9, alpha=0.35)
    trails_c.append([fading_trail(ax, BODY_COLORS[b]) for b in range(3)])
    dots_c.append([ax.plot([], [], "o", color=BODY_COLORS[b], ms=11, mec="white", mew=0.8)[0] for b in range(3)])
time_text_c = axes[0].text(0.03, 0.95, "", transform=axes[0].transAxes, va="top")
fig.text(0.5, 0.02, "same initial condition, same time step dt = %g; grey line = true orbit" % dt,
         ha="center", color=REF_COLOR, fontsize=11)


def update_compare(k):
    artists = []
    for Q, trl, dts in zip(data, trails_c, dots_c):
        for b in range(3):
            update_trail(*trl[b], Q[max(0, k - TRAIL):k + 1, b])
            dts[b].set_data([Q[k, b, 0]], [Q[k, b, 1]])
            artists += [trl[b][0], dts[b]]
    time_text_c.set_text(f"t = {t[k]:.1f}")
    return artists + [time_text_c]


anim_compare = animation.FuncAnimation(fig, update_compare, frames=frames, interval=1000 / FPS, blit=True)
save_anim(anim_compare, "moore_symplectic_vs_euler")

### 3. Stability: two nearby initial conditions

The second trajectory has every position and velocity component perturbed by a random amount of order $10^{-4}$ (the centre of mass and total momentum are kept at zero, so the pair differs only in the *shape* of the orbit, not by a trivial drift).

In [ ]:
rng = np.random.default_rng(0)
eps = 1e-4
q0_p = q0 + eps * rng.standard_normal(q0.shape)
v0_p = v0 + eps * rng.standard_normal(v0.shape)
# remove centre-of-mass offset and net momentum
q0_p -= q0_p.mean(axis=0)
v0_p -= v0_p.mean(axis=0)

_, q_p, _ = symplectic_stromer_verlet(acceleration, q0_p, v0_p, dt, N_i)

# largest distance between corresponding bodies of the two runs
sep = np.max(np.linalg.norm(q - q_p, axis=-1), axis=1)

fig, (ax, ax_s) = plt.subplots(1, 2, figsize=(13, 5), gridspec_kw={"width_ratios": [1.6, 1]})
setup_axes(ax, xlim, ylim, "Two nearby initial conditions")

trails_a = [fading_trail(ax, BODY_COLORS[b]) for b in range(3)]
trails_b = [fading_trail(ax, PERT_COLORS[b], lw=1.6) for b in range(3)]
dots_a = [ax.plot([], [], "o", color=BODY_COLORS[b], ms=12, mec="white", mew=0.8)[0] for b in range(3)]
dots_b = [ax.plot([], [], "o", mfc="none", color=PERT_COLORS[b], ms=18, mew=1.8)[0] for b in range(3)]

ax_s.plot(t, sep, color=REF_COLOR, lw=0.8, alpha=0.3)
(sep_line,) = ax_s.plot([], [], color="#ff8a65", lw=2)
ax_s.set_xlim(0, t[-1])
ax_s.set_ylim(0, sep.max() * 1.15)
ax_s.set_xlabel("t")
ax_s.set_ylabel("max separation between runs")
ax_s.set_title("Separation between the two runs")
ax_s.grid(alpha=0.5)
for s in ("top", "right"):
    ax_s.spines[s].set_visible(False)
fig.tight_layout()


def update_stability(k):
    artists = []
    for b in range(3):
        update_trail(*trails_a[b], q[max(0, k - TRAIL):k + 1, b])
        update_trail(*trails_b[b], q_p[max(0, k - TRAIL):k + 1, b])
        dots_a[b].set_data([q[k, b, 0]], [q[k, b, 1]])
        dots_b[b].set_data([q_p[k, b, 0]], [q_p[k, b, 1]])
        artists += [trails_a[b][0], trails_b[b][0], dots_a[b], dots_b[b]]
    sep_line.set_data(t[:k + 1], sep[:k + 1])
    return artists + [sep_line]


anim_stab = animation.FuncAnimation(fig, update_stability, frames=frames, interval=1000 / FPS, blit=True)
save_anim(anim_stab, "moore_stability")

### 4. The Hamiltonian over time

In [ ]:
H_s = hamiltonian(q, v)
H_e = hamiltonian(q_e, v_e)
H0 = H_s[0]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

ax1.plot(t, H_e, color="#ff8a65", lw=2, label="Explicit Euler")
ax1.plot(t, H_s, color="#4fc3f7", lw=2, label="Symplectic (Störmer–Verlet)")
ax1.axhline(H0, color=REF_COLOR, ls="--", lw=1, label=f"exact $H_0$ = {H0:.4f}")
ax1.set_xlabel("t")
ax1.set_ylabel("H")
ax1.set_title("Hamiltonian, same time step")
ax1.legend(loc="center right")
ax1.grid(alpha=0.5)

ax2.plot(t, H_s - H0, color="#4fc3f7", lw=1.5)
ax2.axhline(0, color=REF_COLOR, ls="--", lw=1)
ax2.set_xlabel("t")
ax2.set_ylabel("$H - H_0$")
ax2.set_title("Symplectic only (zoomed): bounded oscillation")
ax2.ticklabel_format(axis="y", style="sci", scilimits=(0, 0))
ax2.grid(alpha=0.5)

for a in (ax1, ax2):
    for s in ("top", "right"):
        a.spines[s].set_visible(False)

fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "moore_hamiltonian.png"), dpi=200)
plt.show()